# 🕴️ KING2-IMAGE — Stickman Dataset Generation + LoRA Training

خط أنابيب كامل: توليد داتاست ستيكمان اصطناعي (~10,000 صورة) عبر SDXL-Turbo، فلترة جودة تلقائية،
تعليق تلقائي (النص المُستخدم للتوليد = caption)، رفع كداتاست HF، ثم تدريب SDXL LoRA فعلي
ورفعه كملحق داخل `RASHID778/king2-image` (مسار `stickman/`، لا يطغى على الـ LoRA العام الموجود).

لا يوجد داتاست جاهز بحجم كافٍ وترخيص واضح لرسومات ستيكمان — لذلك التوليد الاصطناعي هو المصدر.

In [ ]:
# 1. تسجيل الدخول HuggingFace أولاً — فشل سريع قبل أي عمل
import os
from pathlib import Path

HF_TOKEN = None
_secret_file = Path('/kaggle/input/king2-secrets/hf_token.txt')
if _secret_file.exists():
    HF_TOKEN = _secret_file.read_text().strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, '❌ HF_TOKEN غير متاح'
os.environ['HF_TOKEN'] = HF_TOKEN
print('✅ التوكن موجود')

In [ ]:
# 2. تثبيت المكتبات
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -qU accelerate transformers peft bitsandbytes datasets huggingface_hub Pillow numpy
!pip uninstall -y wandb torchao -q
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py -O train_text_to_image_lora_sdxl.py
!sed -i 's/is_wandb_available()/False/g' train_text_to_image_lora_sdxl.py
!sed -i 's/log_with=args.report_to/log_with=None/g' train_text_to_image_lora_sdxl.py
print('✅ تم التثبيت')

In [ ]:
# 2.5 تسجيل دخول HF — استيراد نظيف بعد الترقية
from huggingface_hub import login, whoami
login(os.environ['HF_TOKEN'])
print('Logged in as:', whoami()['name'])

In [ ]:
# 3. بناء قائمة الأوصاف (prompts) — توليفات منظّمة لتغطية أوضاع/حركات/انفعالات/خلفيات متنوعة
import random

random.seed(42)

STYLE = [
    'simple black and white stick figure drawing',
    'minimalist line art stick figure illustration',
    'clean vector stick figure drawing',
    'cartoon stick figure sketch',
]

SINGLE_ACTIONS = [
    'running', 'jumping', 'walking', 'sitting on a chair', 'standing still', 'waving hello',
    'dancing', 'climbing a ladder', 'riding a bicycle', 'swimming', 'punching', 'kicking',
    'throwing a ball', 'reading a book', 'sleeping', 'crying', 'laughing', 'pointing forward',
    'clapping hands', 'stretching', 'falling down', 'flying like a superhero', 'praying',
    'bowing', 'stretching arms up', 'playing guitar', 'playing basketball', 'cooking',
    'painting on a canvas', 'writing on paper', 'typing on a keyboard', 'driving a car',
    'skateboarding', 'surfing a wave', 'skiing down a hill', 'boxing', 'doing yoga',
    'meditating cross-legged', 'shrugging', 'thinking with hand on chin', 'scratching head',
    'drinking coffee', 'eating food', 'brushing teeth', 'taking a selfie', 'holding an umbrella',
    'holding a sword', 'shooting a bow and arrow', 'playing with a dog', 'holding a balloon',
    'blowing out a candle', 'opening a gift box', 'fishing by a lake', 'gardening',
    'playing chess', 'doing a handstand', 'doing a cartwheel', 'doing a backflip',
    'a karate kick pose', 'saluting', 'texting on a phone', 'climbing a mountain',
    'riding a skateboard down stairs', 'juggling balls', 'playing the drums',
]

MULTI_ACTIONS = [
    'two stick figures shaking hands', 'two stick figures fighting', 'two stick figures hugging',
    'two stick figures high-fiving', 'two stick figures racing each other',
    'two stick figures playing catch with a ball', 'a group of stick figures forming a circle',
    'a team of stick figures playing soccer', 'two stick figures dancing together',
    'a crowd of stick figures cheering',
]

EMOTIONS = [
    'happy expression', 'sad expression', 'angry expression', 'surprised expression',
    'scared expression', 'confused expression', 'excited expression', 'tired expression',
    'proud expression', 'curious expression', 'determined expression', '',
]

BACKGROUNDS = [
    'on a plain white background', 'in a simple outdoor park scene', 'on a stage',
    'in a gym', 'in a city street', 'on a mountain', 'at the beach', 'in a classroom',
    'in space with stars', 'underwater with bubbles', 'in a kitchen',
]

def build_prompt():
    style = random.choice(STYLE)
    emotion = random.choice(EMOTIONS)
    bg = random.choice(BACKGROUNDS)
    if random.random() < 0.15:
        action = random.choice(MULTI_ACTIONS)
        subject = action
    else:
        action = random.choice(SINGLE_ACTIONS)
        subject = f'a stickman {action}'
    parts = [style, 'of', subject]
    if emotion:
        parts.append(f', {emotion}')
    parts.append(f', {bg}')
    return ' '.join(parts).replace(' , ', ', ')

N_TARGET = 10000
N_CANDIDATES = 11500  # margin for quality-filter rejections
prompts = [build_prompt() for _ in range(N_CANDIDATES)]
print(f'✅ عدد الأوصاف المرشّحة: {len(prompts)} — عيّنة:')
for p in prompts[:5]:
    print(' -', p)

In [ ]:
# 4. تحميل SDXL-Turbo — سرعة عالية مناسبة لمحتوى بسيط كالستيكمان (1-4 خطوات كافية)
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.set_progress_bar_config(disable=True)
print('✅ SDXL-Turbo جاهز')

In [ ]:
# 5. التوليد + فلترة جودة تلقائية + تعليق تلقائي (النص المُستخدم = caption)
import json
import numpy as np
from PIL import Image

RAW_DIR = '/kaggle/working/stickman_raw'
os.makedirs(RAW_DIR, exist_ok=True)
GEN_RES = 512  # دقة توليد سريعة؛ تُرفَع لاحقاً إلى 1024 عند التعبئة النهائية
BATCH = 8

def quality_score(img: Image.Image) -> int:
    """تقييم 1-10: يرفض الصور الفارغة/المشوّهة عبر نسبة البكسلات الداكنة (خطوط الستيكمان السوداء)."""
    arr = np.asarray(img.convert('L'), dtype=np.float32)
    dark_ratio = float((arr < 128).mean())
    std = float(arr.std())
    if std < 8:
        return 1  # صورة شبه موحّدة اللون — فاشلة
    if dark_ratio < 0.005:
        return 2  # لا يوجد رسم تقريباً (كل شيء أبيض)
    if dark_ratio > 0.55:
        return 3  # ضجيج/تشويه كامل تقريباً
    # نطاق معقول لخطوط ستيكمان واضحة على خلفية بسيطة
    score = 10
    if not (0.01 <= dark_ratio <= 0.35):
        score -= 3
    if std < 30:
        score -= 2
    return max(1, min(10, score))

meta = []
kept = 0
idx = 0
while kept < N_TARGET and idx < len(prompts):
    batch_prompts = prompts[idx: idx + BATCH]
    idx += BATCH
    try:
        images = pipe(
            prompt=batch_prompts,
            num_inference_steps=2,
            guidance_scale=0.0,
            height=GEN_RES, width=GEN_RES,
        ).images
    except Exception as e:
        print('⚠️ batch failed:', e)
        continue
    for img, prompt in zip(images, batch_prompts):
        if kept >= N_TARGET:
            break
        score = quality_score(img)
        if score < 8:
            continue
        fn = f'{kept:06d}.png'
        img.convert('RGB').save(os.path.join(RAW_DIR, fn))
        clean_caption = prompt.replace('simple black and white stick figure drawing', 'a stickman') \
                               .replace('minimalist line art stick figure illustration', 'a stickman') \
                               .replace('clean vector stick figure drawing', 'a stickman') \
                               .replace('cartoon stick figure sketch', 'a stickman')
        tags = ['stickman', 'stick figure', 'vector', 'minimal', 'black and white']
        meta.append({
            'file_name': fn, 'prompt': prompt, 'caption': clean_caption,
            'tags': tags, 'quality_score': score,
        })
        kept += 1
    if kept % 500 < BATCH:
        print(f'  توليد: {kept}/{N_TARGET}  (استُهلك {idx} وصف من {len(prompts)})')

print(f'✅ تم توليد وقبول {kept} صورة (من أصل {idx} محاولة)')
if kept < N_TARGET:
    print(f'⚠️ لم نصل للهدف {N_TARGET} — نفدت الأوصاف المرشّحة. زد N_CANDIDATES في الخلية 3 وأعد التشغيل من هناك.')

In [ ]:
# 6. تحرير ذاكرة SDXL-Turbo قبل التدريب
del pipe
torch.cuda.empty_cache()
print('✅ تم تحرير الذاكرة')

In [ ]:
# 7. تعبئة الداتاست بالبنية المطلوبة: PNG / RGB / 1024×1024 + captions/ + metadata.jsonl + README
DATASET_DIR = '/kaggle/working/dataset'
IMAGES_DIR = os.path.join(DATASET_DIR, 'images')
CAPTIONS_DIR = os.path.join(DATASET_DIR, 'captions')
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(CAPTIONS_DIR, exist_ok=True)

final_meta = []
for m in meta:
    src = os.path.join(RAW_DIR, m['file_name'])
    img = Image.open(src).convert('RGB').resize((1024, 1024), Image.LANCZOS)
    img.save(os.path.join(IMAGES_DIR, m['file_name']))
    with open(os.path.join(CAPTIONS_DIR, m['file_name'].replace('.png', '.txt')), 'w', encoding='utf-8') as f:
        f.write(m['caption'])
    final_meta.append({
        'file_name': f"images/{m['file_name']}",
        'prompt': m['caption'],
        'tags': m['tags'],
        'quality_score': m['quality_score'],
    })

with open(os.path.join(DATASET_DIR, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
    for m in final_meta:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

README = f'''# KING2 Stickman Dataset

Synthetic dataset of {len(final_meta)} stick-figure images generated with SDXL-Turbo,
for training the KING2-IMAGE SDXL LoRA (stickman style/pose adapter).

No suitable licensed real-world dataset of illustrated stick figures existed at this
scale, so images were synthesized with a large combinatorial prompt template covering
~70 single-figure actions, ~10 multi-figure interactions, 11 emotions, and 11 backgrounds.
Each image's generation prompt doubles as its caption (captions/ + metadata.jsonl `prompt`).

Quality filter: heuristic (dark-pixel ratio + pixel std-dev) rejecting blank/noise
generations; a quality_score (8-10) is recorded per image in metadata.jsonl.

## Structure
```
dataset/
├── images/       1024x1024 RGB PNG
├── captions/     one .txt per image (same caption as metadata.jsonl)
├── metadata.jsonl
└── README.md
```
'''
with open(os.path.join(DATASET_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(README)

print(f'✅ الداتاست جاهز: {len(final_meta)} صورة في {DATASET_DIR}')

In [ ]:
# 8. رفع الداتاست إلى HuggingFace
from huggingface_hub import HfApi, create_repo

DATASET_REPO = 'RASHID778/king2-stickman-dataset'
create_repo(DATASET_REPO, repo_type='dataset', exist_ok=True)
HfApi().upload_folder(
    folder_path=DATASET_DIR,
    repo_id=DATASET_REPO, repo_type='dataset',
    commit_message=f'KING2 stickman synthetic dataset — {len(final_meta)} images',
)
print(f'✅ تم الرفع: https://huggingface.co/datasets/{DATASET_REPO}')

In [ ]:
# 9. إعداد بيانات التدريب بصيغة imagefolder (caption_column='prompt' من metadata.jsonl)
TRAIN_DIR = '/kaggle/working/king2_stickman_train'
os.makedirs(TRAIN_DIR, exist_ok=True)
import shutil
for m in final_meta:
    shutil.copy(os.path.join(DATASET_DIR, m['file_name']), os.path.join(TRAIN_DIR, os.path.basename(m['file_name'])))
with open(os.path.join(TRAIN_DIR, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
    for m in final_meta:
        f.write(json.dumps({'file_name': os.path.basename(m['file_name']), 'prompt': m['prompt']}, ensure_ascii=False) + '\n')
print(f'✅ بيانات التدريب جاهزة: {len(final_meta)} صورة في {TRAIN_DIR}')

In [ ]:
# 10. إعداد accelerate — GPU واحد
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='fp16')
print('✅ accelerate config')

In [ ]:
# 11. التدريب — SDXL LoRA على داتاست الستيكمان (نفس الإعدادات المُثبَتة على T4، خطوات مُعدَّلة لحجم البيانات الأكبر)
!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/kaggle/working/king2_stickman_train" \
  --caption_column="prompt" \
  --resolution=768 --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --mixed_precision="fp16" \
  --max_train_steps=3500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" --lr_warmup_steps=0 \
  --rank=16 \
  --checkpointing_steps=1000 \
  --seed=42 \
  --output_dir="/kaggle/working/king2-stickman-lora" \
  --report_to="tensorboard" 2>&1 | tee /kaggle/working/train_stickman.log

_w = '/kaggle/working/king2-stickman-lora/pytorch_lora_weights.safetensors'
if os.path.exists(_w):
    print('\n✅✅ التدريب نجح:', _w)
else:
    print('\n❌❌ التدريب فشل — آخر 40 سطر:')
    !tail -n 40 /kaggle/working/train_stickman.log
    raise SystemExit('training failed')

In [ ]:
# 12. اختبار النموذج — توليد صورة ستيكمان
from diffusers import DiffusionPipeline

test_pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16
).to('cuda')
test_pipe.load_lora_weights('/kaggle/working/king2-stickman-lora', weight_name='pytorch_lora_weights.safetensors')
image = test_pipe('a stickman doing a backflip, excited expression, on a plain white background',
                   num_inference_steps=30, guidance_scale=7.0).images[0]
image.save('/kaggle/working/stickman_test.png')
del test_pipe
torch.cuda.empty_cache()
print('✅ تم توليد صورة اختبارية')
image

In [ ]:
# 13. رفع LoRA الستيكمان داخل نفس مستودع RASHID778/king2-image (مسار stickman/ — لا يطغى على الـ LoRA العام)
MODEL_REPO = 'RASHID778/king2-image'
STICKMAN_DIR = '/kaggle/working/king2-stickman-lora'

STICKMAN_CARD_APPEND = f'''

## 🕴️ Stickman adapter

A second LoRA in this repo, trained on a synthetic {len(final_meta)}-image stickman
dataset (`{DATASET_REPO}`), for stick-figure poses/actions/illustrations.

```python
pipe.load_lora_weights("{MODEL_REPO}", weight_name="stickman/pytorch_lora_weights.safetensors")
image = pipe("a stickman running, happy expression, on a plain white background").images[0]
```
'''
with open(os.path.join(STICKMAN_DIR, 'README_APPEND.md'), 'w', encoding='utf-8') as f:
    f.write(STICKMAN_CARD_APPEND)

api = HfApi()
for fn in ['pytorch_lora_weights.safetensors']:
    api.upload_file(
        path_or_fileobj=os.path.join(STICKMAN_DIR, fn),
        path_in_repo=f'stickman/{fn}',
        repo_id=MODEL_REPO, repo_type='model',
        commit_message='Add stickman-style LoRA adapter (synthetic dataset)',
    )
print(f'✅ تم الرفع: https://huggingface.co/{MODEL_REPO}/tree/main/stickman')
print('⚠️ أضف محتوى README_APPEND.md يدويًا لملف README.md الرئيسي في المستودع (خلية 13 كتبته في مجلد stickman/ محليًا فقط).')